# Conversational AI with Ministral 3-14B (PyTorch)

Build a conversational chatbot using Mistral AI's Ministral 3-14B model with PyTorch and HuggingFace Transformers.

**What you'll learn:**
- Load HuggingFace credentials from `.env` using python-dotenv
- Configure 4-bit quantization with bitsandbytes for memory efficiency
- Load and run a 14B parameter model with PyTorch/Transformers
- Build a conversational chatbot with context retention
- Tune generation parameters for optimal responses

## 1. Import Libraries and Setup

In [ ]:
import torch
import os
import warnings
from pathlib import Path

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv()

# Configure HuggingFace cache to persistent storage
# This ensures models persist across container restarts
MODELS_DIR = Path(os.getenv("MODELS_DIR", "/workspace/models"))
HF_CACHE_DIR = MODELS_DIR / "huggingface"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "hub")

# HuggingFace transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig
)

# Set random seed for reproducibility
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Models directory: {MODELS_DIR}")
print(f"HuggingFace cache: {HF_CACHE_DIR}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Load Environment Variables

We securely load the HuggingFace API token from a `.env` file. This token is required for downloading gated models.

**Setup:** Create a `.env` file in your notebook directory with:
```
HF_TOKEN=hf_your_token_here
```

Get your token from: https://huggingface.co/settings/tokens

In [ ]:
# Load environment variables from .env file
env_path = Path(".env")
if env_path.exists():
    load_dotenv(env_path)
    print("\u2705 Loaded .env file")
else:
    print("\u26a0\ufe0f  No .env file found. Create one with HF_TOKEN=your_token")
    print("   Get your token: https://huggingface.co/settings/tokens")

# Verify HuggingFace token
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")

if hf_token:
    # Mask token for display
    masked = f"{hf_token[:8]}...{hf_token[-4:]}" if len(hf_token) > 12 else "***"
    print(f"\u2705 HuggingFace token loaded: {masked}")
else:
    print("\u26a0\ufe0f  No HuggingFace token found.")
    print("   Some models may not be accessible without authentication.")

## 3. Configure Device and Memory

For large language models, proper memory management is crucial. We'll configure:
- **Device selection**: GPU if available, with CPU fallback
- **Data type**: `bfloat16` for efficient memory usage on modern GPUs
- **Memory monitoring**: Track VRAM usage during model loading

The Ministral 3-14B model requires approximately:
- ~8-10 GB VRAM in 4-bit quantization
- ~14-16 GB VRAM in 8-bit quantization
- ~28 GB VRAM in bfloat16 (no quantization)

In [ ]:
# Device configuration
if torch.cuda.is_available():
    device = torch.device("cuda")
    # Use bfloat16 for modern GPUs (Ampere and newer)
    compute_capability = torch.cuda.get_device_capability()
    if compute_capability[0] >= 8:  # Ampere or newer
        dtype = torch.bfloat16
        print(f"\u2705 Using bfloat16 (GPU compute capability {compute_capability[0]}.{compute_capability[1]})")
    else:
        dtype = torch.float16
        print(f"\u26a0\ufe0f  Using float16 (GPU compute capability {compute_capability[0]}.{compute_capability[1]})")
else:
    device = torch.device("cpu")
    dtype = torch.float32
    print("\u26a0\ufe0f  No GPU detected. Using CPU (this will be slow for LLMs)")

print(f"\nDevice: {device}")
print(f"Data type: {dtype}")

# Memory monitoring helper
def print_gpu_memory():
    """Display current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved, {total:.1f} GB total")

print_gpu_memory()

## 4. Configure 4-bit Quantization

We use `bitsandbytes` for 4-bit quantization with NF4 (Normal Float 4) data type. This reduces the model's memory footprint while maintaining quality.

| Config | VRAM | Quality |
|--------|------|---------|  
| 4-bit NF4 | ~8-10 GB | Good (recommended for 16GB) |
| 8-bit | ~14-16 GB | Better |
| BF16 | ~28 GB | Best |

In [ ]:
# 4-bit quantization config for 16GB VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,  # Nested quantization for extra memory savings
    bnb_4bit_quant_type="nf4"         # Normal Float 4 - optimized for neural networks
)

print("\u2705 Quantization configuration:")
print(f"   - 4-bit quantization: Enabled")
print(f"   - Compute dtype: {dtype}")
print(f"   - Double quantization: Enabled")
print(f"   - Quantization type: NF4")

## 5. Load Model with PyTorch/Transformers

We'll load the **Ministral 3-14B-Instruct** model from Mistral AI. This is an instruction-tuned model optimized for chat and conversational tasks.

**Model properties:**
- 14 billion parameters
- 256k token context window
- Apache 2.0 license

**Note:** First run will download ~28GB of model weights (cached for future use).

In [ ]:
# Model configuration
MODEL_ID = "mistralai/Ministral-3-14B-Instruct-2512"

print(f"Loading model: {MODEL_ID}")
print(f"Cache directory: {HF_CACHE_DIR}")
print("This may take a few minutes on first run (downloading ~28GB)...")
print()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
    cache_dir=HF_CACHE_DIR
)

# Ensure pad token is set (required for batch generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    torch_dtype=dtype,
    device_map="auto",              # Automatically distribute across available devices
    quantization_config=quantization_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True,         # Reduce RAM usage during loading
    cache_dir=HF_CACHE_DIR
)

print(f"✅ Model loaded successfully")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print()
print_gpu_memory()

## 6. Configure Generation Parameters

Generation parameters control how the model produces text:

| Parameter | Description | Our Value |
|-----------|-------------|----------|
| `temperature` | Randomness (higher = more creative) | 0.7 |
| `top_p` | Nucleus sampling threshold | 0.9 |
| `top_k` | Limit vocabulary selection | 50 |
| `max_new_tokens` | Maximum response length | 512 |
| `repetition_penalty` | Discourage repetition | 1.1 |

In [ ]:
# Generation configuration
generation_config = GenerationConfig(
    max_new_tokens=512,          # Maximum tokens to generate
    temperature=0.7,             # Creativity (0.1=focused, 1.0=creative)
    top_p=0.9,                   # Nucleus sampling
    top_k=50,                    # Vocabulary limitation
    repetition_penalty=1.1,      # Discourage repetition
    do_sample=True,              # Enable sampling (required for temperature)
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print("\u2705 Generation parameters configured:")
print(f"   max_new_tokens: {generation_config.max_new_tokens}")
print(f"   temperature: {generation_config.temperature}")
print(f"   top_p: {generation_config.top_p}")
print(f"   top_k: {generation_config.top_k}")
print(f"   repetition_penalty: {generation_config.repetition_penalty}")

## 7. Build ChatBot Class

We'll create a `ChatBot` class that:
- Maintains conversation history across messages
- Formats messages using the model's chat template
- Generates contextual responses with PyTorch
- Provides memory management utilities

In [ ]:
import time

class ChatBot:
    """Conversational chatbot using Ministral 3-14B with PyTorch."""
    
    def __init__(self, model, tokenizer, generation_config, system_prompt=None):
        self.model = model
        self.tokenizer = tokenizer
        self.generation_config = generation_config
        self.conversation_history = []
        
        # Default system prompt
        self.system_prompt = system_prompt or (
            "You are a helpful, harmless, and honest AI assistant. "
            "Provide clear, accurate, and thoughtful responses."
        )
        
    def reset(self):
        """Clear conversation history."""
        self.conversation_history = []
        print("\U0001f504 Conversation history cleared")
        
    def chat(self, user_message, verbose=True):
        """Send a message and get a response."""
        start_time = time.time()
        
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Build messages list with system prompt
        messages = [
            {"role": "system", "content": self.system_prompt}
        ] + self.conversation_history
        
        # Apply chat template
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096  # Context window limit for efficiency
        ).to(self.model.device)
        
        # Generate response with PyTorch
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                generation_config=self.generation_config
            )
        
        # Decode response (skip input tokens)
        response = self.tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        # Add assistant response to history
        self.conversation_history.append({
            "role": "assistant",
            "content": response
        })
        
        elapsed = time.time() - start_time
        
        if verbose:
            print(f"\U0001f916 {response}")
            print(f"\n   [{elapsed:.1f}s]")
            
        return response
    
    def get_history(self):
        """Return conversation history."""
        return self.conversation_history.copy()
    
    def get_token_count(self):
        """Estimate tokens in conversation history."""
        if not self.conversation_history:
            return 0
        all_text = " ".join([m["content"] for m in self.conversation_history])
        return len(self.tokenizer.encode(all_text))

# Initialize chatbot
chatbot = ChatBot(model, tokenizer, generation_config)
print("\u2705 ChatBot initialized and ready!")

## 8. Interactive Conversation

Let's test the chatbot with a conversation! The chatbot maintains context across multiple messages.

In [ ]:
print("=" * 60)
print("Starting conversation...")
print("=" * 60)
print()

# First message
print("\U0001f464 User: What is machine learning?")
chatbot.chat("What is machine learning?")
print()

In [ ]:
# Follow-up question (tests context retention)
print("\U0001f464 User: Can you give me a simple example?")
chatbot.chat("Can you give me a simple example?")
print()

In [ ]:
# Another follow-up
print("\U0001f464 User: How is this different from traditional programming?")
chatbot.chat("How is this different from traditional programming?")

## 9. Interactive Chat Loop (Optional)

Run this cell for a fully interactive chat experience. Type your messages and press Enter.

**Commands:**
- Type `quit` or `exit` to end the conversation
- Type `reset` to clear history and start fresh

**Note:** This cell requires user input and will block until you type `quit`.

In [ ]:
def interactive_chat(chatbot):
    """Run interactive chat loop."""
    print("=" * 60)
    print("Interactive Chat Mode")
    print("Commands: 'quit'/'exit' to end, 'reset' to clear history")
    print("=" * 60)
    print()
    
    while True:
        try:
            user_input = input("\U0001f464 You: ").strip()
            
            if not user_input:
                continue
                
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\n\U0001f44b Goodbye!")
                break
                
            if user_input.lower() == 'reset':
                chatbot.reset()
                continue
            
            print()
            chatbot.chat(user_input)
            print()
            
        except KeyboardInterrupt:
            print("\n\n\U0001f44b Chat interrupted. Goodbye!")
            break

# Uncomment the line below to start interactive mode:
# interactive_chat(chatbot)

print("\U0001f4a1 Tip: Uncomment the line above to start interactive chat mode")

## 10. Custom System Prompts

You can customize the chatbot's personality and behavior using system prompts. Here are some examples:

In [ ]:
# Example: Create a specialized coding assistant
coding_assistant = ChatBot(
    model, 
    tokenizer, 
    generation_config,
    system_prompt=(
        "You are an expert Python programmer and software engineer. "
        "Provide clear, well-documented code examples. "
        "Always explain your code and suggest best practices. "
        "If asked about other languages, you can help but prefer Python examples."
    )
)

print("\U0001f40d Coding Assistant")
print("-" * 40)
print("\U0001f464 User: Write a function to check if a number is prime")
coding_assistant.chat("Write a function to check if a number is prime")

In [ ]:
# Example: Create a creative writing assistant
creative_writer = ChatBot(
    model, 
    tokenizer, 
    generation_config,
    system_prompt=(
        "You are a creative writing assistant with a flair for vivid imagery "
        "and compelling narratives. Help users craft engaging stories, poems, "
        "and creative content. Be imaginative and inspiring."
    )
)

print("\U0001f3a8 Creative Writer")
print("-" * 40)
print("\U0001f464 User: Write a haiku about artificial intelligence")
creative_writer.chat("Write a haiku about artificial intelligence")

## 11. Memory Management and Cleanup

Large language models consume significant GPU memory. Here's how to monitor and clean up resources.

In [ ]:
# Memory status
print("Current GPU Memory Usage:")
print_gpu_memory()

# Show conversation history size
history = chatbot.get_history()
print(f"\nConversation history: {len(history)} messages")

# Token count for context
token_count = chatbot.get_token_count()
print(f"Approximate tokens in history: {token_count:,}")

# Context usage estimate
max_context = 4096  # Our working context limit
print(f"Context usage: {token_count / max_context * 100:.1f}% of {max_context:,} token limit")

In [ ]:
# Optional: Clear model from memory (uncomment if needed)
# This is useful if you need to free GPU memory for other tasks

# del model
# del tokenizer
# del chatbot
# torch.cuda.empty_cache()
# print("\U0001f9f9 Model cleared from GPU memory")

print("\U0001f4a1 Tip: Uncomment the lines above to clear model from memory")

## Summary

**What we accomplished:**
- \u2705 Loaded HuggingFace credentials securely from `.env`
- \u2705 Configured 4-bit NF4 quantization for memory efficiency
- \u2705 Loaded the 14B parameter Ministral model with PyTorch
- \u2705 Built a conversational chatbot with history management
- \u2705 Explored generation parameters and custom system prompts

**Key takeaways:**
1. **Memory efficiency**: Use 4-bit quantization (bitsandbytes) to fit large models on consumer GPUs
2. **Chat templates**: Always use `apply_chat_template()` for proper message formatting
3. **Context management**: Monitor token count to avoid context window limits
4. **Reproducibility**: Set `torch.manual_seed()` for consistent behavior

**Next steps:**
- Experiment with different generation parameters (temperature, top_p)
- Try 8-bit quantization if you have more VRAM
- Add streaming output for real-time response display
- Implement RAG (Retrieval-Augmented Generation) for knowledge grounding
- Explore function calling capabilities

**Resources:**
- [Mistral AI Documentation](https://docs.mistral.ai/)
- [HuggingFace Transformers](https://huggingface.co/docs/transformers/)
- [Ministral Model Card](https://huggingface.co/mistralai/Ministral-3-14B-Instruct-2512)
- [BitsAndBytes Quantization](https://huggingface.co/docs/transformers/quantization/bitsandbytes)
- [Generation Strategies](https://huggingface.co/docs/transformers/generation_strategies)